# Rule: **build_electricity_demand_base**

This rule distributes the national electricity demand time series over the buses of the base (simplified) network, producing an hourly demand time series per bus.

The national hourly demand (`resources/electricity_demand.csv`) is *upsampled* to the onshore regions of the base network using a spatial distribution key. In the default PyPSA-Eur path, the key combines GDP and population at NUTS3 level (weights from `load: distribution_key`), the JRC Energy Atlas raster, and, for Great Britain, the DESNZ consumption statistics per local authority.

In PyPSA-Spain, when `pypsa_spain: electricity_demand: enable: true`, an alternative path (`upsample_load_vPyPSA_Spain`) is used instead for Spain: the demand is built from NUTS3 profiles disaggregated by economic sector, and the annual total is scaled to the configured `annual_value` (in TWh).

**Outputs**

- resources/`electricity_demand_base_s.nc`


In [ ]:
######################################## Parameters

### Run
name = ''
prefix = ''

### Spatial domain 'ES' or 'EU' (for maps domain)
spatial_domain = 'ES'


In [ ]:
##### Import packages
import xarray as xr
import pandas as pd
import numpy as np
import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import os
import sys
from matplotlib.colors import LogNorm


##### Import local functions
sys.path.append(os.path.abspath(os.path.join('..')))
import functions as xp


##### Read params.yaml
params = xp.read_params('../params.yaml')


##### Ignore warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning)


##### Region files
region_tag = f'base_s'
gdf_regions_onshore, _ = xp.load_regions(
    params,
    prefix=prefix,
    name=name,
    region_tag=region_tag,
)


## `electricity_demand_base_s.nc`

Load the dataset and show its structure.

In [ ]:
file = f'electricity_demand_base_s.nc'
path = f'{params["rootpath"]}/resources/{prefix}/{name}/'

ds = xr.open_dataset(path+file)

ds


The dataset contains a single variable, `electricity demand (MW)`, indexed by `time` (hourly timestamps) and `bus` (buses of the base network). Buses are identified by their OSM id, and match the `name` field of `regions_onshore_base_s.geojson`.

In [ ]:
#################### Derived parameters
variable = 'electricity demand (MW)'

da = ds[variable]

### Demand aggregated over buses [MW]
total_system = da.sum(dim='bus').to_series()

### Annual demand per bus [TWh]
annual_by_bus = (da.sum(dim='time').to_series() / 1e6).sort_values(ascending=False)
annual_by_bus.name = 'annual_demand'


#### Summary

What are the main figures of the demand time series?

If the PyPSA-Spain path is enabled, the total annual demand should match `pypsa_spain: electricity_demand: annual_value` in the configuration file.

In [ ]:
#################### Derived parameters
hours = len(total_system)
total_twh = float(da.sum()) / 1e6
peak_mw = float(total_system.max())
min_mw = float(total_system.min())
mean_mw = float(total_system.mean())


#################### Summary
summary = pd.Series({
    'buses': da.sizes['bus'],
    'snapshots (hours)': hours,
    'period': f'{total_system.index[0]:%Y-%m-%d} to {total_system.index[-1]:%Y-%m-%d}',
    'total annual demand [TWh]': f'{total_twh:,.2f}',
    'peak demand [MW]': f'{peak_mw:,.0f}',
    'minimum demand [MW]': f'{min_mw:,.0f}',
    'mean demand [MW]': f'{mean_mw:,.0f}',
    'load factor [-]': f'{mean_mw/peak_mw:.3f}',
    'NaN values': int(da.isnull().sum()),
    'negative values': int((da < 0).sum()),
})

summary.to_frame('value')


#### Time series

How does the aggregated hourly demand (summed over buses) look along the year?

In [ ]:
#################### Parameters

### Define period
start = '2013-01-01'
end = '2013-12-31'



#################### Figure
fig_size = [12,5]
fig, ax = plt.subplots(figsize=fig_size)

serie = total_system.loc[start:end]

ax.plot(serie.index, serie.values, linewidth=.5, alpha=.9, color='tab:blue')

ax.grid(True, linestyle='--', alpha=0.5)
ax.set_ylabel('MW')
ax.set_title(f'Hourly electricity demand aggregated over buses. Total: {total_twh:,.2f} TWh')


How is the demand distributed along the year and along the day? The heatmap below shows the aggregated demand for every hour of the year (day of the year x hour of the day), revealing the seasonal and daily patterns.

In [ ]:
#################### Derived parameters
heat = pd.DataFrame({
    'day': total_system.index.dayofyear,
    'hour': total_system.index.hour,
    'value': total_system.values,
    }).pivot_table(index='hour', columns='day', values='value')



#################### Figure
fig_size = [12,5]
fig, ax = plt.subplots(figsize=fig_size)

pcm = ax.pcolormesh(heat.columns, heat.index, heat.values, cmap='YlOrRd', shading='auto')

fig.colorbar(pcm, ax=ax, label='MW')

ax.set_xlabel('day of the year')
ax.set_ylabel('hour of the day')
ax.set_yticks(range(0, 24, 3))
ax.set_title('Aggregated electricity demand [MW]')


#### Maps

Plot the annual electricity demand per region in two ways: the absolute value [TWh], and the demand density [GWh/km2], i.e. the annual demand divided by the area of the region. Areas are computed on the equal-area projection EPSG:3035 (ETRS89-LAEA), the same one the rule uses to redistribute attributes between geometries.

The absolute demand mostly reflects how large each region is, while the density isolates where the demand is actually concentrated. The density spans several orders of magnitude, so it is drawn on a logarithmic colour scale; regions without demand cannot be represented on that scale and are shown in grey.

In [ ]:
#################### Local params
params_local = {}
params_local['vmin'] = ''           # Annual demand [TWh]. Leave empty for automatic value
params_local['vmax'] = ''           # Annual demand [TWh]. Leave empty for automatic value
params_local['vmin_density'] = ''   # Demand density [GWh/km2]. Leave empty for automatic value
params_local['vmax_density'] = ''   # Demand density [GWh/km2]. Leave empty for automatic value



#################### Data
gdf = gdf_regions_onshore.merge(annual_by_bus, left_on='name', right_index=True, how='left')

### Area of each region on an equal-area projection [km2]
gdf['area'] = gdf.to_crs(epsg=3035).area / 1e6

### Demand density [GWh/km2]
gdf['density'] = 1e3 * gdf['annual_demand'] / gdf['area']

### Regions without demand cannot be placed on a log scale
gdf.loc[gdf['density'] <= 0, 'density'] = np.nan



#################### Figure
fig_size = [14,6]
crs = ccrs.PlateCarree()

fig, axes = plt.subplots(1, 2, figsize=fig_size, subplot_kw={'projection': crs})


### Fix params_local
vmin = params_local['vmin'] if params_local['vmin'] != '' else gdf['annual_demand'].min()
vmax = params_local['vmax'] if params_local['vmax'] != '' else gdf['annual_demand'].max()
vmin_density = params_local['vmin_density'] if params_local['vmin_density'] != '' else gdf['density'].min()
vmax_density = params_local['vmax_density'] if params_local['vmax_density'] != '' else gdf['density'].max()


### Add annual demand at regions
xp.map_add_features(axes[0], params['map_add_features'])

gdf.plot(ax=axes[0], column='annual_demand',
         cmap='Blues', edgecolor='black', lw=0.2,
         vmin=vmin, vmax=vmax,
         legend=True, legend_kwds={'fraction': 0.030, 'pad': 0.02})

axes[0].set_extent(params[f'boundaries_onshore_{spatial_domain}'])

total = gdf['annual_demand'].sum()
axes[0].set_title(f'Annual electricity demand [TWh]. Total: {total:,.2f} TWh')


### Add demand density at regions
xp.map_add_features(axes[1], params['map_add_features'])

gdf.plot(ax=axes[1], column='density',
         cmap='Blues', edgecolor='black', lw=0.2,
         norm=LogNorm(vmin=vmin_density, vmax=vmax_density),
         legend=True, legend_kwds={'fraction': 0.030, 'pad': 0.02},
         missing_kwds={'color': 'lightgrey'})

axes[1].set_extent(params[f'boundaries_onshore_{spatial_domain}'])

mean_density = 1e3 * total / gdf['area'].sum()
axes[1].set_title(f'Demand density [GWh/km2]. Mean: {mean_density:,.2f} GWh/km2')


plt.tight_layout()
